In [ ]:
from pathlib import Path

import polars as pl
import xgboost as xgb

import wandb

In [ ]:
REGISTRY_PATH = "wandb-registry-model/solar-flare-xgboost"
INPUT_FEATURES = [
    "xrsb_flux",
    "lag_15",
    "lag_60",
    "lag_120",
    "lag_1440",
    "roll_max_720",
    "roll_std_720",
    "roll_mean_720",
    "deriv_1_5",
    "deriv_2_5",
    "roll_c_class_cross_720",
]

In [ ]:
api = wandb.Api(
    overrides={
        "entity": "jaybrandon-hochschule-luzern",
        "project": "solar-flux-ml-pipeline",
    }
)

In [ ]:
artifact_path = REGISTRY_PATH + ":production"

In [ ]:
artifact: wandb.Artifact = api.artifact(artifact_path)

In [ ]:
artifact_dir = Path(artifact.download())

bst = xgb.Booster(model_file=artifact_dir / "model.json")

In [ ]:
artifact.logged_by().summary

In [ ]:
artifact.name

In [ ]:
artifact.version

In [ ]:
artifact.aliases

In [ ]:
artifact.created_at

In [ ]:
fs = "gs://mlops-solar-flux_online_fs"

In [ ]:
df = pl.read_parquet(f"{fs}/*.parquet")
df

In [ ]:
df.with_columns(pl.col(pl.Float64) * 10000000)

In [ ]:
dinf = xgb.DMatrix(df[INPUT_FEATURES])

In [ ]:
pred = bst.predict(dinf)[0]

In [ ]:
float(pred)

In [ ]:
(float(pred) / 1000000) > 1e-5